In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
# Load dataset from root folder (has upazila column)
df = pd.read_csv("../datasetNew/mergeDataset.csv")
print(df.shape)
df.head()

(65983, 27)


,datetime,year,month,day,hour,weekday,temperature,humidity,rainfall,wind_speed,...,upazila,area_type,substation_id,feeder_id,transformer_age,transformer_capacity,outage_history,maintenance_due,population_density,industrial_load_ratio
0,2021-01-01 00:00:00,2021,1,1,0,4,14.9,91.0,0.0,6.6,...,Teknaf,Rural,SS_009,FDR_09,19,200,8,No,1633,0.05
1,2021-01-01 00:00:00,2021,1,1,0,4,12.5,98.0,0.0,7.2,...,Banshkhali,Rural,SS_049,FDR_06,24,350,0,Yes,818,0.06
2,2021-01-01 00:00:00,2021,1,1,0,4,16.2,95.0,0.0,5.6,...,Kaptai,Rural,SS_020,FDR_11,24,300,9,Yes,793,0.17
3,2021-01-01 00:00:00,2021,1,1,0,4,14.0,99.0,0.0,4.7,...,Matlab South,Rural,SS_030,FDR_10,12,400,7,Yes,1404,0.28
4,2021-01-01 00:00:00,2021,1,1,0,4,11.8,92.0,0.0,6.7,...,Noakhali Sadar,Urban,SS_022,FDR_01,24,350,1,No,6729,0.49


In [3]:
print(df.isnull().sum())

datetime                 0
year                     0
month                    0
day                      0
hour                     0
weekday                  0
temperature              0
humidity                 0
rainfall                 0
wind_speed               0
weather_state            0
electricity_demand       0
renewable_generation     0
transformer_load         0
risk_score               0
risk_level               0
district                 0
upazila                  0
area_type                0
substation_id            0
feeder_id                0
transformer_age          0
transformer_capacity     0
outage_history           0
maintenance_due          0
population_density       0
industrial_load_ratio    0
dtype: int64


In [4]:
print(df["risk_level"].value_counts())
print(df["risk_level"].value_counts(normalize=True))

risk_level
Low       47211
High       9448
Medium     9324
Name: count, dtype: int64
risk_level
Low       0.715502
High      0.143188
Medium    0.141309
Name: proportion, dtype: float64


In [5]:
# 1. Convert datetime and sort chronologically per feeder
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values(by=['substation_id', 'feeder_id', 'datetime']).copy()

# 1.5 Feature Engineering
cap_smooth = df['transformer_capacity'] + 1e-5
df['load_utilization'] = df['transformer_load'] / cap_smooth
df['demand_utilization'] = df['electricity_demand'] / cap_smooth
df['renewable_ratio'] = df['renewable_generation'] / (df['electricity_demand'] + 1e-5)
df['thi'] = df['temperature'] + 0.55 * (1 - df['humidity']/100.0) * (df['temperature'] - 14.5)
df['wind_temp_interaction'] = df['wind_speed'] * df['temperature']
df['is_peak_hour'] = df['hour'].apply(lambda h: 1 if 18 <= h <= 22 else 0)
df['is_weekend'] = df['weekday'].apply(lambda w: 1 if w >= 5 else 0).copy()

# 1.5 Feature Engineering
cap_smooth = df['transformer_capacity'] + 1e-5
df['load_utilization'] = df['transformer_load'] / cap_smooth
df['demand_utilization'] = df['electricity_demand'] / cap_smooth
df['renewable_ratio'] = df['renewable_generation'] / (df['electricity_demand'] + 1e-5)
df['thi'] = df['temperature'] + 0.55 * (1 - df['humidity']/100.0) * (df['temperature'] - 14.5)
df['wind_temp_interaction'] = df['wind_speed'] * df['temperature']
df['is_peak_hour'] = df['hour'].apply(lambda h: 1 if 18 <= h <= 22 else 0)
df['is_weekend'] = df['weekday'].apply(lambda w: 1 if w >= 5 else 0)

# 2. Shift risk_level by -1 to forecast the NEXT step
df['future_risk_level'] = df.groupby(['substation_id', 'feeder_id'])['risk_level'].shift(-1)

# 3. Drop rows that have no future data (the last timestamp for each feeder)
df = df.dropna(subset=['future_risk_level'])

# 4. Overwrite risk_level with the future_risk_level
df['risk_level'] = df['future_risk_level']

# 5. Split data into chronological Train (80%) and Test (20%) per feeder
train_dfs = []
test_dfs = []
for name, group in df.groupby(['substation_id', 'feeder_id']):
    n = len(group)
    split_idx = int(n * 0.8)
    train_dfs.append(group.iloc[:split_idx])
    test_dfs.append(group.iloc[split_idx:])

train_df = pd.concat(train_dfs)
test_df = pd.concat(test_dfs)

print("Train DF shape:", train_df.shape)
print("Test DF shape:", test_df.shape)


Train DF shape: (51587, 35)
Test DF shape: (13396, 35)


In [6]:
categorical_cols = [
    "weather_state",
    "district",
    "upazila",
    "area_type",
    "substation_id",
    "feeder_id",
    "maintenance_due"
]

encoders = {} 

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    # Gracefully map test set using train encoder classes, handling unseen classes if any
    test_df[col] = test_df[col].map(lambda s: s if s in le.classes_ else le.classes_[0])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

import joblib
os.makedirs("../models", exist_ok=True)
joblib.dump(encoders, "../models/categorical_encoders.pkl")
print("Saved categorical_encoders.pkl")

Saved categorical_encoders.pkl


In [7]:
target_encoder = LabelEncoder()
train_df["risk_level"] = target_encoder.fit_transform(train_df["risk_level"])
test_df["risk_level"] = target_encoder.transform(test_df["risk_level"])
joblib.dump(target_encoder, "../models/target_encoder.pkl")
print("Saved target_encoder.pkl")

Saved target_encoder.pkl


In [8]:
# Drop columns not used as features
drop_cols = [
    "datetime",
    "risk_score",
    "year",
    "month",
    "day",
    "future_risk_level"
]

X_train = train_df.drop(columns=drop_cols + ["risk_level"], errors='ignore')
y_train = train_df["risk_level"]
X_test = test_df.drop(columns=drop_cols + ["risk_level"], errors='ignore')
y_test = test_df["risk_level"]

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape, "y_test shape:", y_test.shape)


X_train shape: (51587, 28) y_train shape: (51587,)
X_test shape: (13396, 28) y_test shape: (13396,)


In [9]:
os.makedirs("../dataset", exist_ok=True)
# Save chronological split unscaled data for tree models
np.save("../dataset/X_train.npy", X_train)
np.save("../dataset/y_train_full.npy", y_train)
np.save("../dataset/X_test.npy", X_test)
np.save("../dataset/y_test_full.npy", y_test)
print("Unscaled chronological splits saved successfully.")


Unscaled chronological splits saved successfully.


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)


X_train_scaled shape: (51587, 28)
X_test_scaled shape: (13396, 28)


In [11]:
joblib.dump(scaler, "../models/scaler.pkl")
np.save("../dataset/X_train_scaled.npy", X_train_scaled)
np.save("../dataset/X_test_scaled.npy", X_test_scaled)
print("Scaler and scaled arrays saved successfully!")


Scaler and scaled arrays saved successfully!


In [12]:
print("Feature columns:")
print(list(X_train.columns))


Feature columns:
['hour', 'weekday', 'temperature', 'humidity', 'rainfall', 'wind_speed', 'weather_state', 'electricity_demand', 'renewable_generation', 'transformer_load', 'district', 'upazila', 'area_type', 'substation_id', 'feeder_id', 'transformer_age', 'transformer_capacity', 'outage_history', 'maintenance_due', 'population_density', 'industrial_load_ratio', 'load_utilization', 'demand_utilization', 'renewable_ratio', 'thi', 'wind_temp_interaction', 'is_peak_hour', 'is_weekend']


In [13]:
TIME_STEPS = 5

# Train sequence generation (within each feeder group)
X_train_3D = []
y_train_seq = []
for name, group in X_train.groupby(['substation_id', 'feeder_id']):
    group_X = X_train_scaled_df.loc[group.index].values
    group_y = y_train.loc[group.index].values
    for i in range(len(group_X) - TIME_STEPS):
        X_train_3D.append(group_X[i : i + TIME_STEPS])
        y_train_seq.append(group_y[i + TIME_STEPS])

X_train_3D = np.array(X_train_3D)
y_train_seq = np.array(y_train_seq)
X_train_2D = X_train_3D.reshape(X_train_3D.shape[0], -1)

# Test sequence generation (within each feeder group)
X_test_3D = []
y_test_seq = []
for name, group in X_test.groupby(['substation_id', 'feeder_id']):
    group_X = X_test_scaled_df.loc[group.index].values
    group_y = y_test.loc[group.index].values
    for i in range(len(group_X) - TIME_STEPS):
        X_test_3D.append(group_X[i : i + TIME_STEPS])
        y_test_seq.append(group_y[i + TIME_STEPS])

X_test_3D = np.array(X_test_3D)
y_test_seq = np.array(y_test_seq)
X_test_2D = X_test_3D.reshape(X_test_3D.shape[0], -1)

print("X_train_3D Shape:", X_train_3D.shape)
print("X_train_2D Shape:", X_train_2D.shape)
print("y_train Shape:", y_train_seq.shape)
print("X_test_3D Shape:", X_test_3D.shape)
print("X_test_2D Shape:", X_test_2D.shape)
print("y_test Shape:", y_test_seq.shape)

np.save("../dataset/X_train_3D.npy", X_train_3D)
np.save("../dataset/X_train_2D.npy", X_train_2D)
np.save("../dataset/y_train.npy", y_train_seq)
np.save("../dataset/X_test_3D.npy", X_test_3D)
np.save("../dataset/X_test_2D.npy", X_test_2D)
np.save("../dataset/y_test.npy", y_test_seq)
print("Chronological train/test sequence datasets saved successfully!")

X_train_3D Shape: (46606, 5, 28)
X_train_2D Shape: (46606, 140)
y_train Shape: (46606,)
X_test_3D Shape: (10027, 5, 28)
X_test_2D Shape: (10027, 140)
y_test Shape: (10027,)
Chronological train/test sequence datasets saved successfully!
